# Módulo 4 — Análise Exploratória de Dados (EDA)
### Curso Introdutório de Python para Ciência de Dados
**Disciplina:** T326 - Ciência de Dados | UNIFOR  
**Dataset:** Brazilian Cities — 5.578 municípios brasileiros

---

## 📚 O que você vai aprender neste módulo

| Seção | Conteúdo |
|-------|----------|
| 4.1 | O que é EDA e por que é essencial |
| 4.2 | Definição do problema e perguntas norteadoras |
| 4.3 | Exploração inicial dos dados |
| 4.4 | Análise univariada — distribuições |
| 4.5 | Análise bivariada — relações entre variáveis |
| 4.6 | Análise categórica — padrões por grupos |
| 4.7 | Storytelling com dados — narrativa analítica |
| 4.8 | Conclusões e comunicação de insights |
| 4.9 | Exercícios |

> 🔍 **EDA** não é apenas explorar — é fazer as **perguntas certas** e deixar os dados responderem.

---
## Configurações e Carregamento dos Dados

In [ ]:
# ── Importações ─────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings, io

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi':       120,
    'figure.facecolor': 'white',
    'axes.facecolor':   '#F8F9FA',
    'axes.grid':        True,
    'grid.alpha':       0.4,
    'font.family':      'DejaVu Sans',
})
sns.set_palette('Set2')

print(f"✅ pandas {pd.__version__} | numpy {np.__version__}")
print(f"✅ matplotlib {plt.matplotlib.__version__} | seaborn {sns.__version__}")

In [ ]:
# ── Carregamento e preparação do dataset ─────────────────────────
def carregar_brazilian_cities(caminho):
    with open(caminho, 'r', encoding='utf-8', newline='') as f:
        raw = f.read()
    linhas = raw.split('\r\n')
    def fix(l):
        if l.startswith('"') and l.endswith('"'):
            l = l[1:-1]
        return l.replace('""', '"')
    conteudo = '\n'.join([fix(l) for l in linhas if l.strip()])
    return pd.read_csv(io.StringIO(conteudo))

df = carregar_brazilian_cities('../../modulo-2/datasets/brazilian_city.csv')

df = df.rename(columns={
    'IDHM Ranking 2010': 'IDHM_Ranking',
    'IBGE_CROP_PRODUCTION_$': 'IBGE_CROP_PROD',
    'WAL-MART': 'WALMART', 'IBGE_1-4': 'IBGE_1a4',
    'IBGE_5-9': 'IBGE_5a9', 'IBGE_10-14': 'IBGE_10a14',
    'IBGE_15-59': 'IBGE_15a59', 'IBGE_60+': 'IBGE_60mais',
})

df['POP_FINAL']  = df['IBGE_RES_POP'].where(df['IBGE_RES_POP'] > 0, df['ESTIMATED_POP'])
df['DENSIDADE']  = np.where(df['AREA'] > 0, (df['POP_FINAL'] / df['AREA']).round(2), np.nan)
df['TIPO']       = df['CAPITAL'].map({1: 'Capital', 0: 'Interior'})
df['IDHM_FAIXA'] = pd.cut(df['IDHM'],
    bins=[0, 0.499, 0.599, 0.699, 0.799, 1.0],
    labels=['Muito Baixo', 'Baixo', 'Médio', 'Alto', 'Muito Alto'])

regioes = {
    'Norte':        ['AM','PA','RR','RO','AC','AP','TO'],
    'Nordeste':     ['MA','PI','CE','RN','PB','PE','AL','SE','BA'],
    'Centro-Oeste': ['MT','MS','GO','DF'],
    'Sudeste':      ['SP','RJ','MG','ES'],
    'Sul':          ['PR','SC','RS']
}
estado_regiao    = {uf: reg for reg, ufs in regioes.items() for uf in ufs}
df['REGIAO']     = df['STATE'].map(estado_regiao)

print(f"✅ Dataset pronto: {df.shape[0]:,} municípios × {df.shape[1]} variáveis")

---
## 4.1 O que é EDA e por que é essencial?

**Análise Exploratória de Dados (EDA)** é o processo de investigar um dataset para:
- Entender sua estrutura e qualidade
- Descobrir padrões, tendências e anomalias
- Formular e testar hipóteses
- Preparar os dados para modelagem

### O Ciclo da EDA

```
1. Definir o Problema
       ↓
2. Exploração Inicial  →  shape, dtypes, nulos, duplicatas
       ↓
3. Análise Univariada  →  distribuição de cada variável
       ↓
4. Análise Bivariada   →  relações entre duas variáveis
       ↓
5. Análise Multivariada → padrões com 3+ variáveis
       ↓
6. Storytelling        →  narrativa com insights
       ↓
7. Conclusões          →  responder as perguntas iniciais
```

### Ferramentas por etapa

| Etapa | Ferramentas Python |
|-------|-------------------|
| Inspeção | `.info()`, `.describe()`, `.isnull()` |
| Univariada | `hist()`, `boxplot()`, `value_counts()` |
| Bivariada | `scatter()`, `heatmap()`, `groupby()` |
| Multivariada | `pairplot()`, `facetgrid()`, `pivot_table()` |
| Storytelling | `subplots()`, anotações, paletas coerentes |

---
## 4.2 Definição do Problema e Perguntas Norteadoras

> 🎯 **Uma boa EDA começa com boas perguntas.**

### Problema Central
> *"O que determina o nível de desenvolvimento humano (IDHM) de um município brasileiro, e como ele varia entre regiões, estados e categorias urbanas?"*

### Perguntas Norteadoras

| # | Pergunta | Tipo de análise |
|---|----------|----------------|
| P1 | Como o IDHM varia entre as 5 regiões do Brasil? | Categórica |
| P2 | Quais variáveis têm maior correlação com o IDHM? | Bivariada |
| P3 | Qual o grau de desigualdade entre municípios do mesmo estado? | Univariada |
| P4 | Cidades mais urbanizadas têm IDHM sistematicamente mais alto? | Categórica |
| P5 | Qual componente do IDHM (Renda, Longevidade, Educação) é gargalo por região? | Multivariada |
| P6 | Municípios maiores são necessariamente mais desenvolvidos? | Bivariada |

### Como estruturar as perguntas

```python
# Boas perguntas têm:
# ✅ Uma variável de interesse clara (ex: IDHM)
# ✅ Um recorte ou comparação (ex: por região, por categoria)
# ✅ Uma hipótese implícita a ser testada
# ❌ "O dataset é interessante?" — vago demais
# ❌ "Mostre tudo sobre o IDHM" — sem foco
```

---
## 4.3 Exploração Inicial dos Dados

In [ ]:
# ── 4.3.1 Estrutura e tipos do dataset ──────────────────────────
print("=" * 55)
print(f" DATASET: {df.shape[0]:,} linhas × {df.shape[1]} colunas")
print("=" * 55)

print("\n📌 Primeiras linhas:")
display(df[['CITY','STATE','REGIAO','IDHM','GDP_CAPITA','POP_FINAL','AREA']].head())

print("\n📌 Tipos de dados:")
print(df.dtypes.value_counts().to_string())

In [ ]:
# ── 4.3.2 Qualidade dos dados: nulos e duplicatas ────────────────
nulos = df.isnull().sum()
nulos_pct = (nulos / len(df) * 100).round(1)
resumo_nulos = pd.DataFrame({'Nulos': nulos, '%': nulos_pct})
resumo_nulos = resumo_nulos[resumo_nulos['Nulos'] > 0].sort_values('%', ascending=False)

print("📌 Variáveis com valores ausentes:")
print(resumo_nulos.head(15).to_string())

duplicatas = df.duplicated().sum()
print(f"\n📌 Duplicatas: {duplicatas}")
print(f"📌 Municípios únicos: {df['CITY'].nunique():,}")
print(f"📌 Estados únicos: {df['STATE'].nunique()}")

In [ ]:
# ── 4.3.3 Estatísticas descritivas das variáveis-chave ───────────
vars_chave = ['IDHM', 'IDHM_Renda', 'IDHM_Longevidade', 'IDHM_Educacao',
              'GDP_CAPITA', 'POP_FINAL', 'AREA', 'DENSIDADE']

desc = df[vars_chave].describe().T
desc.columns = ['n', 'Média', 'Desvio', 'Mín', 'Q1', 'Mediana', 'Q3', 'Máx']
desc['n'] = desc['n'].astype(int)
display(desc.round(3))

# Coeficiente de variação (CV) — mede dispersão relativa
cv = (df[vars_chave].std() / df[vars_chave].mean() * 100).round(1)
print("\n📌 Coeficiente de Variação (%) — quanto maior, mais heterogêneo:")
for col, val in cv.sort_values(ascending=False).items():
    barra = '█' * int(val / 5)
    print(f"  {col:<22} {val:>6.1f}%  {barra}")

---
## 4.4 Análise Univariada — Distribuições

Analisar **uma variável por vez** para entender sua distribuição, presença de outliers e assimetria.

In [ ]:
# ── 4.4.1 Distribuições das componentes do IDHM ──────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Distribuições das Componentes do IDHM — Brasil (5.578 municípios)',
             fontsize=14, fontweight='bold', y=1.01)

variaveis = [
    ('IDHM',             'IDHM Geral',    '#4C72B0'),
    ('IDHM_Renda',       'IDHM Renda',    '#55A868'),
    ('IDHM_Longevidade', 'IDHM Longevidade', '#C44E52'),
    ('IDHM_Educacao',    'IDHM Educação', '#8172B2'),
]

for ax, (col, titulo, cor) in zip(axes.flatten(), variaveis):
    dados = df[col].dropna()
    ax.hist(dados, bins=35, color=cor, edgecolor='white', alpha=0.85)
    ax.axvline(dados.mean(),   color='black',  linestyle='--', linewidth=1.5,
               label=f'Média = {dados.mean():.3f}')
    ax.axvline(dados.median(), color='orange', linestyle='-.',  linewidth=1.5,
               label=f'Mediana = {dados.median():.3f}')
    ax.set_title(titulo, fontsize=12, fontweight='bold')
    ax.set_xlabel('Valor')
    ax.set_ylabel('Municípios')
    ax.legend(fontsize=9)
    # Assimetria
    skew = dados.skew()
    ax.text(0.98, 0.92, f'Assimetria: {skew:.2f}',
            transform=ax.transAxes, ha='right', fontsize=8, color='gray')

plt.tight_layout()
plt.show()

print("💡 Insight: IDHM_Educação tem a maior dispersão — há municípios com educação muito baixa.")

In [ ]:
# ── 4.4.2 Detecção de outliers com boxplot ───────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Análise de Outliers — GDP per Capita e População', fontsize=14, fontweight='bold')

# GDP per Capita — escala original (com outliers visíveis)
gdp_valido = df[df['GDP_CAPITA'] > 0]['GDP_CAPITA']
axes[0].boxplot(gdp_valido, vert=True, patch_artist=True,
                boxprops=dict(facecolor='#4C72B0', alpha=0.7),
                medianprops=dict(color='orange', linewidth=2))
axes[0].set_title('PIB per Capita — Escala Original')
axes[0].set_ylabel('R$')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
n_outliers = ((gdp_valido > gdp_valido.quantile(0.75) + 1.5*(gdp_valido.quantile(0.75) - gdp_valido.quantile(0.25)))).sum()
axes[0].text(1.05, gdp_valido.max()*0.85, f'{n_outliers} outliers\nsuperiores', fontsize=9, color='red')

# PIB per Capita — escala log (distribuição mais clara)
axes[1].boxplot(np.log10(gdp_valido), vert=True, patch_artist=True,
                boxprops=dict(facecolor='#55A868', alpha=0.7),
                medianprops=dict(color='orange', linewidth=2))
axes[1].set_title('PIB per Capita — Escala Logarítmica')
axes[1].set_ylabel('log₁₀(PIB per Capita)')
ticks = [3, 3.5, 4, 4.5, 5]
axes[1].set_yticks(ticks)
axes[1].set_yticklabels([f'R${10**t:,.0f}' for t in ticks])

plt.tight_layout()
plt.show()

print("💡 Insight: A escala log revela que o PIB per capita tem distribuição mais simétrica")
print("   quando transformado — útil para modelagem e comparações visuais.")

In [ ]:
# ── 4.4.3 Contagens categóricas ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Distribuição por Categorias', fontsize=14, fontweight='bold')

# Municípios por faixa de IDHM
contagem_faixa = df['IDHM_FAIXA'].value_counts().sort_index()
cores_faixa = ['#D32F2F', '#FF7043', '#FDD835', '#66BB6A', '#1565C0']
bars = axes[0].bar(contagem_faixa.index.astype(str), contagem_faixa.values,
                   color=cores_faixa, edgecolor='white')
for bar, val in zip(bars, contagem_faixa.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                 f'{val:,}\n({val/len(df)*100:.1f}%)', ha='center', va='bottom', fontsize=9)
axes[0].set_title('Municípios por Faixa de IDHM')
axes[0].set_ylabel('Número de Municípios')
axes[0].set_xlabel('Faixa de IDHM')
axes[0].tick_params(axis='x', rotation=15)

# Municípios por região
contagem_regiao = df['REGIAO'].value_counts().reindex(
    ['Norte', 'Nordeste', 'Centro-Oeste', 'Sudeste', 'Sul'])
cores_regiao = ['#E07B54', '#E0C354', '#54A0E0', '#54E089', '#9B54E0']
bars2 = axes[1].bar(contagem_regiao.index, contagem_regiao.values,
                    color=cores_regiao, edgecolor='white')
for bar, val in zip(bars2, contagem_regiao.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 15,
                 f'{val:,}', ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[1].set_title('Municípios por Região')
axes[1].set_ylabel('Número de Municípios')
axes[1].set_xlabel('Região')

plt.tight_layout()
plt.show()

---
## 4.5 Análise Bivariada — Relações entre Variáveis

Analisar como **duas variáveis se relacionam** — correlação, associação, causa e efeito.

In [ ]:
# ── 4.5.1 Matriz de correlação — variáveis numéricas ─────────────
vars_corr = [
    'IDHM', 'IDHM_Renda', 'IDHM_Longevidade', 'IDHM_Educacao',
    'GDP_CAPITA', 'DENSIDADE', 'POP_FINAL', 'PAY_TV', 'Cars'
]
corr = df[vars_corr].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True,
            linewidths=0.5, cbar_kws={'label': 'Correlação de Pearson'}, ax=ax)
ax.set_title('Matriz de Correlação — Variáveis Socioeconômicas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Ranking das correlações com IDHM
print("\n📌 Correlação com IDHM (ranking):")
corr_idhm = corr['IDHM'].drop('IDHM').sort_values(ascending=False)
for var, val in corr_idhm.items():
    sinal = '▲' if val > 0 else '▼'
    barra = '█' * int(abs(val) * 20)
    print(f"  {sinal} {var:<22} {val:+.3f}  {barra}")

In [ ]:
# ── 4.5.2 Scatter plots — IDHM × principais fatores ─────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Relações Bivariadas com o IDHM', fontsize=14, fontweight='bold')

ordem_reg = ['Norte', 'Nordeste', 'Centro-Oeste', 'Sudeste', 'Sul']
palette_reg = dict(zip(ordem_reg, ['#E07B54', '#E0C354', '#54A0E0', '#54E089', '#9B54E0']))

# P1: IDHM × GDP_CAPITA
df_plot = df[(df['GDP_CAPITA'] > 0) & (df['GDP_CAPITA'] < 200_000)].dropna(subset=['IDHM','REGIAO'])
for reg, grp in df_plot.groupby('REGIAO'):
    axes[0].scatter(grp['IDHM'], grp['GDP_CAPITA'],
                    alpha=0.25, s=15, color=palette_reg.get(reg,'gray'), label=reg)
z = np.polyfit(df_plot['IDHM'].dropna(), df_plot['GDP_CAPITA'].dropna(), 1)
xr = np.linspace(df_plot['IDHM'].min(), df_plot['IDHM'].max(), 100)
axes[0].plot(xr, np.poly1d(z)(xr), 'k--', linewidth=2, alpha=0.8)
axes[0].set_title('IDHM × PIB per Capita')
axes[0].set_xlabel('IDHM')
axes[0].set_ylabel('PIB per Capita (R$)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
corr_gdp = df_plot[['IDHM','GDP_CAPITA']].corr().iloc[0,1]
axes[0].text(0.05, 0.95, f'r = {corr_gdp:.3f}', transform=axes[0].transAxes,
             fontsize=10, fontweight='bold', color='black')

# P2: IDHM × DENSIDADE (log scale)
df_dens = df[(df['DENSIDADE'] > 0) & (df['DENSIDADE'] < 10_000)].dropna(subset=['IDHM','REGIAO'])
for reg, grp in df_dens.groupby('REGIAO'):
    axes[1].scatter(grp['IDHM'], np.log10(grp['DENSIDADE']),
                    alpha=0.25, s=15, color=palette_reg.get(reg,'gray'))
axes[1].set_title('IDHM × Densidade (log)')
axes[1].set_xlabel('IDHM')
axes[1].set_ylabel('log₁₀(hab/km²)')
corr_dens = df_dens[['IDHM']].assign(log_dens=np.log10(df_dens['DENSIDADE'])).corr().iloc[0,1]
axes[1].text(0.05, 0.95, f'r = {corr_dens:.3f}', transform=axes[1].transAxes,
             fontsize=10, fontweight='bold')

# P3: IDHM_Educação × IDHM_Renda
df_comp = df.dropna(subset=['IDHM_Educacao','IDHM_Renda','REGIAO'])
for reg, grp in df_comp.groupby('REGIAO'):
    axes[2].scatter(grp['IDHM_Renda'], grp['IDHM_Educacao'],
                    alpha=0.3, s=15, color=palette_reg.get(reg,'gray'), label=reg)
axes[2].set_title('Renda × Educação (componentes IDHM)')
axes[2].set_xlabel('IDHM Renda')
axes[2].set_ylabel('IDHM Educação')
axes[2].legend(fontsize=7, markerscale=2)
corr_comp = df_comp[['IDHM_Renda','IDHM_Educacao']].corr().iloc[0,1]
axes[2].text(0.05, 0.95, f'r = {corr_comp:.3f}', transform=axes[2].transAxes,
             fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ── 4.5.3 Agrupamentos — média por estado ────────────────────────
resumo_estado = df.groupby('STATE').agg(
    IDHM_medio     = ('IDHM',            'mean'),
    GDP_medio      = ('GDP_CAPITA',       'mean'),
    n_municipios   = ('CITY',             'count'),
    IDHM_desvio    = ('IDHM',            'std'),
    REGIAO         = ('REGIAO',           'first'),
).reset_index()

fig, ax = plt.subplots(figsize=(12, 7))
for reg in ordem_reg:
    grp = resumo_estado[resumo_estado['REGIAO'] == reg]
    ax.scatter(grp['GDP_medio'], grp['IDHM_medio'],
               s=grp['n_municipios'] * 0.8,
               color=palette_reg[reg], alpha=0.85, label=reg,
               edgecolors='white', linewidth=0.5)
    for _, row in grp.iterrows():
        ax.annotate(row['STATE'],
                    (row['GDP_medio'], row['IDHM_medio']),
                    fontsize=7, ha='center', va='bottom', color='#333333')

ax.set_title('PIB per Capita × IDHM por Estado\n(tamanho do círculo = número de municípios)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('PIB per Capita Médio do Estado (R$)', fontsize=11)
ax.set_ylabel('IDHM Médio do Estado', fontsize=11)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R${x/1000:.0f}k'))
ax.legend(fontsize=9, title='Região')
plt.tight_layout()
plt.show()

print("\n💡 Insight: Há uma relação positiva entre PIB e IDHM, mas não linear.")
print("   Alguns estados do Sul têm IDHM elevado com PIB moderado.")

---
## 4.6 Análise Categórica — Padrões por Grupos

In [ ]:
# ── 4.6.1 Pivot table — Faixa IDHM × Região ─────────────────────
pivot = pd.crosstab(df['REGIAO'], df['IDHM_FAIXA'], normalize='index') * 100
pivot = pivot.reindex(ordem_reg)

fig, ax = plt.subplots(figsize=(13, 6))
cores_faixa = ['#D32F2F', '#FF7043', '#FDD835', '#66BB6A', '#1565C0']

bottom = np.zeros(len(pivot))
for col, cor in zip(pivot.columns, cores_faixa):
    valores = pivot[col].values
    bars = ax.bar(pivot.index, valores, bottom=bottom, color=cor, label=str(col), edgecolor='white')
    for bar, val in zip(bars, valores):
        if val > 4:
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_y() + bar.get_height()/2,
                    f'{val:.0f}%', ha='center', va='center',
                    fontsize=9, fontweight='bold', color='white')
    bottom += valores

ax.set_title('Composição por Faixa de IDHM em cada Região (%)',
             fontsize=14, fontweight='bold')
ax.set_ylabel('% de Municípios', fontsize=12)
ax.set_ylim(0, 100)
ax.legend(title='Faixa IDHM', loc='upper right', framealpha=0.9)
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
plt.tight_layout()
plt.show()

print("💡 Norte e Nordeste concentram a maioria dos municípios nas faixas Baixo e Muito Baixo.")
print("   Sul e Sudeste têm proporção expressiva nas faixas Alto e Muito Alto.")

In [ ]:
# ── 4.6.2 Componentes IDHM por região — gargalo regional ─────────
comp_regiao = df.groupby('REGIAO')[['IDHM_Renda','IDHM_Longevidade','IDHM_Educacao']].mean()
comp_regiao.columns = ['Renda', 'Longevidade', 'Educação']
comp_regiao = comp_regiao.reindex(ordem_reg)

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(ordem_reg))
width = 0.25
cores_comp = ['#4C72B0', '#55A868', '#C44E52']

for i, (col, cor) in enumerate(zip(comp_regiao.columns, cores_comp)):
    bars = ax.bar(x + i * width, comp_regiao[col], width, label=col,
                  color=cor, edgecolor='white', alpha=0.9)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.004,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)

ax.set_title('Componentes do IDHM por Região — Onde está o gargalo?',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Região', fontsize=12)
ax.set_ylabel('Valor Médio do Componente', fontsize=12)
ax.set_xticks(x + width)
ax.set_xticklabels(ordem_reg)
ax.set_ylim(0.5, 0.9)
ax.legend(title='Componente', fontsize=10)
ax.axhline(0.7, color='gray', linestyle=':', alpha=0.6, label='Referência 0.700')
plt.tight_layout()
plt.show()

# Identificar gargalo por região
print("\n📌 Gargalo por região (componente com menor valor médio):")
for reg in ordem_reg:
    row = comp_regiao.loc[reg]
    gargalo = row.idxmin()
    print(f"  {reg:<15}: {gargalo} ({row[gargalo]:.3f})")

In [ ]:
# ── 4.6.3 Capitais vs Interior ───────────────────────────────────
df_tipo = df.dropna(subset=['IDHM','TIPO'])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Capitais vs Interior — Desenvolvimento Humano', fontsize=14, fontweight='bold')

# Boxplot por tipo
sns.boxplot(data=df_tipo, x='TIPO', y='IDHM', palette=['#E06C75','#4C72B0'],
            width=0.5, fliersize=3, ax=axes[0])
axes[0].set_title('Distribuição do IDHM')
axes[0].set_xlabel('')
axes[0].set_ylabel('IDHM')
for tipo, dados in df_tipo.groupby('TIPO'):
    n = len(dados)
    med = dados['IDHM'].median()
    axes[0].text(['Capital','Interior'].index(tipo), med + 0.01,
                 f'Mediana: {med:.3f}\n(n={n})', ha='center', fontsize=9)

# GDP por tipo
df_gdp = df_tipo[df_tipo['GDP_CAPITA'] > 0]
sns.boxplot(data=df_gdp, x='TIPO', y='GDP_CAPITA',
            palette=['#E06C75','#4C72B0'], width=0.5, fliersize=3, ax=axes[1])
axes[1].set_title('Distribuição do PIB per Capita')
axes[1].set_xlabel('')
axes[1].set_ylabel('PIB per Capita (R$)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R${x/1000:.0f}k'))

plt.tight_layout()
plt.show()

# Comparação quantitativa
cap = df_tipo[df_tipo['TIPO'] == 'Capital']['IDHM']
int_ = df_tipo[df_tipo['TIPO'] == 'Interior']['IDHM']
print(f"📌 Capitais: média IDHM = {cap.mean():.3f} | Interior: {int_.mean():.3f}")
print(f"📌 Diferença: +{(cap.mean() - int_.mean()):.3f} nas capitais")

---
## 4.7 Storytelling com Dados — Narrativa Analítica

**Storytelling com dados** é a habilidade de transformar análises em narrativas claras e convincentes.

### Estrutura de uma boa narrativa analítica:
1. **Contexto** — qual é a situação?
2. **Conflito/Tensão** — o que é surpreendente ou problemático?
3. **Resolução** — o que os dados revelam?
4. **Chamada para ação** — o que devemos fazer com isso?

> 🎯 **Regra de ouro:** Cada visualização deve responder uma pergunta específica.

In [ ]:
# ── 4.7.1 Dashboard narrativo: A desigualdade do IDHM ────────────
fig = plt.figure(figsize=(16, 10))
gs  = fig.add_gridspec(2, 3, hspace=0.45, wspace=0.35)

fig.suptitle('Brasil: A Desigualdade no Desenvolvimento Humano Municipal',
             fontsize=16, fontweight='bold', y=1.01)

# Painel 1 (topo-esquerda): Distribuição nacional
ax1 = fig.add_subplot(gs[0, 0])
ax1.hist(df['IDHM'].dropna(), bins=40, color='#4C72B0', edgecolor='white', alpha=0.85)
ax1.axvline(df['IDHM'].mean(), color='red',    linestyle='--', label=f"Média={df['IDHM'].mean():.3f}")
ax1.axvline(df['IDHM'].median(), color='green', linestyle='-.', label=f"Mediana={df['IDHM'].median():.3f}")
ax1.set_title('① Distribuição Nacional', fontweight='bold')
ax1.set_xlabel('IDHM'); ax1.set_ylabel('Municípios')
ax1.legend(fontsize=8)

# Painel 2 (topo-centro): Boxplot por região
ax2 = fig.add_subplot(gs[0, 1])
sns.boxplot(data=df.dropna(subset=['IDHM','REGIAO']), x='REGIAO', y='IDHM',
            order=ordem_reg,
            palette=['#E07B54','#E0C354','#54A0E0','#54E089','#9B54E0'],
            width=0.55, fliersize=2, ax=ax2)
ax2.set_title('② IDHM por Região', fontweight='bold')
ax2.set_xlabel(''); ax2.set_ylabel('IDHM')
ax2.tick_params(axis='x', rotation=20)
ax2.axhline(df['IDHM'].median(), color='gray', linestyle=':', alpha=0.7)

# Painel 3 (topo-direita): Top e Bottom 10 municípios
ax3 = fig.add_subplot(gs[0, 2])
top10    = df.nlargest(5, 'IDHM')[['CITY','STATE','IDHM']]
bottom10 = df.nsmallest(5, 'IDHM')[['CITY','STATE','IDHM']]
combined = pd.concat([top10, bottom10])
labels   = [f"{r['CITY'][:12]}/{r['STATE']}" for _, r in combined.iterrows()]
cores_tb = ['#1565C0'] * 5 + ['#D32F2F'] * 5
ax3.barh(range(len(combined)), combined['IDHM'].values, color=cores_tb, edgecolor='white')
ax3.set_yticks(range(len(combined)))
ax3.set_yticklabels(labels, fontsize=8)
ax3.set_title('③ Top 5 e Bottom 5', fontweight='bold')
ax3.set_xlabel('IDHM')
ax3.axvline(df['IDHM'].mean(), color='gray', linestyle='--', alpha=0.6)

# Painel 4 (baixo-esquerda): Componentes gargalo
ax4 = fig.add_subplot(gs[1, 0])
comp_nac = df[['IDHM_Renda','IDHM_Longevidade','IDHM_Educacao']].mean()
comp_nac.index = ['Renda','Longevidade','Educação']
bars_c = ax4.bar(comp_nac.index, comp_nac.values,
                 color=['#4C72B0','#55A868','#C44E52'], edgecolor='white')
for bar in bars_c:
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{bar.get_height():.3f}', ha='center', fontsize=9, fontweight='bold')
ax4.set_title('④ Componentes Nacionais', fontweight='bold')
ax4.set_ylabel('Média Nacional')
ax4.set_ylim(0.5, 0.85)
ax4.axhline(0.7, color='red', linestyle='--', alpha=0.5, label='Limiar 0.700')
ax4.legend(fontsize=8)

# Painel 5 (baixo-centro): IDHM × GDP scatter
ax5 = fig.add_subplot(gs[1, 1])
df_s = df[(df['GDP_CAPITA'] > 0) & (df['GDP_CAPITA'] < 150_000)].dropna(subset=['IDHM','REGIAO'])
for reg in ordem_reg:
    g = df_s[df_s['REGIAO'] == reg]
    ax5.scatter(g['IDHM'], g['GDP_CAPITA'], alpha=0.2, s=10,
                color={'Norte':'#E07B54','Nordeste':'#E0C354','Centro-Oeste':'#54A0E0',
                       'Sudeste':'#54E089','Sul':'#9B54E0'}[reg])
ax5.set_title('⑤ IDHM × PIB per Capita', fontweight='bold')
ax5.set_xlabel('IDHM'); ax5.set_ylabel('PIB per Capita')
ax5.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))

# Painel 6 (baixo-direita): % municípios acima de 0.7 por região
ax6 = fig.add_subplot(gs[1, 2])
pct_alto = df.groupby('REGIAO').apply(
    lambda g: (g['IDHM'] >= 0.7).sum() / len(g) * 100
).reindex(ordem_reg)
cores6 = ['#E07B54' if v < 30 else '#FDD835' if v < 60 else '#66BB6A' for v in pct_alto]
bars6 = ax6.bar(pct_alto.index, pct_alto.values, color=cores6, edgecolor='white')
for bar, val in zip(bars6, pct_alto.values):
    ax6.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{val:.0f}%', ha='center', fontsize=10, fontweight='bold')
ax6.set_title('⑥ % municípios com IDHM ≥ 0.7', fontweight='bold')
ax6.set_ylabel('% Municípios')
ax6.tick_params(axis='x', rotation=20)
ax6.yaxis.set_major_formatter(mticker.PercentFormatter())

plt.show()
print("\n📖 NARRATIVA:")
print("  O Brasil apresenta extrema desigualdade no desenvolvimento humano:")
print(f"  - O Norte tem {pct_alto['Norte']:.0f}% dos municípios com IDHM ≥ 0.7")
print(f"  - O Sul tem {pct_alto['Sul']:.0f}% dos municípios com IDHM ≥ 0.7")
print(f"  - O componente Educação ({comp_nac['Educação']:.3f}) é o maior gargalo nacional.")

---
## 4.8 Conclusões e Comunicação de Insights

### O que aprendemos com esta EDA?

| Pergunta | Resposta |
|----------|----------|
| **P1 - Variação regional** | Norte e Nordeste têm IDHM significativamente menor; Sul e Sudeste lideram |
| **P2 - Fatores associados** | PIB per capita e infraestrutura (TV a cabo, telefone) correlacionam-se fortemente com IDHM |
| **P3 - Desigualdade** | Todos os estados possuem alta dispersão interna; o problema não é só entre estados |
| **P4 - Urbanização** | Capitais têm IDHM sistematicamente superior, mas existem municípios rurais de alto desempenho |
| **P5 - Gargalo** | Educação é o componente com menor valor em todas as regiões |
| **P6 - Tamanho** | Não há relação direta entre área e desenvolvimento — municípios pequenos podem ter alto IDHM |

### Boas práticas de comunicação de insights

```python
# ✅ Insight bem estruturado:
# 1. Afirmação clara: "Municípios do Sul têm 3x mais chance de IDHM alto"
# 2. Evidência: "68% dos municípios sulistas têm IDHM ≥ 0.7 vs 14% no Norte"
# 3. Contexto: "Reflexo de diferenças históricas em educação e infraestrutura"
# 4. Implicação: "Políticas nacionais precisam de recorte regional"

# ❌ Insight fraco:
# "O Sul tem IDHM maior que o Norte" — sem evidência quantitativa ou contexto
```

### Próximos passos após a EDA
- **Modelagem preditiva:** Regressão para prever IDHM com base nos fatores identificados
- **Análise temporal:** Comparar evolução do IDHM ao longo dos Censos
- **Análise espacial:** Mapas coropléticos para visualizar distribuição geográfica

---
## 4.9 Exercícios Práticos — Módulo 4

> O **gabarito** está no arquivo `gabarito_exercicios_modulo4.ipynb`.

---

### Exercício 1 — Exploração inicial
Calcule e exiba um resumo das seguintes métricas para o dataset:
- Número de municípios por região (com percentual do total)
- As 3 variáveis com maior quantidade de valores ausentes
- Coeficiente de variação do IDHM por estado (qual estado tem mais heterogeneidade interna?)

```python
# Seu código aqui
```

---

### Exercício 2 — Análise univariada
Crie um painel com **4 subgráficos** mostrando a distribuição de:
`IDHM`, `IDHM_Educacao`, `GDP_CAPITA` (log10) e `DENSIDADE` (log10).
Para cada um, adicione linhas de média e mediana.

```python
# Seu código aqui
```

---

### Exercício 3 — Análise bivariada
Calcule a correlação de Pearson entre `IDHM` e as seguintes variáveis:
`GDP_CAPITA`, `IDHM_Educacao`, `IDHM_Renda`, `IDHM_Longevidade`, `PAY_TV`, `Cars`, `DENSIDADE`.

Crie um gráfico de barras horizontais com os valores de correlação, colorindo barras positivas de azul e negativas de vermelho.

```python
# Seu código aqui
```

---

### Exercício 4 — Análise categórica
Crie uma tabela pivot com `REGIAO` nas linhas e `IDHM_FAIXA` nas colunas, mostrando a **quantidade** (não percentual) de municípios em cada combinação. Em seguida, plote um heatmap desta tabela com `seaborn`.

```python
# Seu código aqui
```

---

### Exercício 5 — Mini-storytelling
Crie um dashboard com **3 painéis** que contem uma história coerente sobre **desigualdade entre estados**. Use pelo menos um boxplot, um gráfico de barras e um scatter plot. Escreva 3 insights com base no que você observou.

```python
# Seu código aqui
```

In [ ]:
# Espaço para suas respostas

# Exercício 1

# Exercício 2

# Exercício 3

# Exercício 4

# Exercício 5

---
## 📌 Resumo — Fluxo de uma EDA em Python

```python
# 1. Carregar e inspecionar
df = pd.read_csv('dados.csv')
df.info()                          # tipos e nulos
df.describe()                      # estatísticas descritivas
df.isnull().sum()                  # contagem de nulos por coluna
df.duplicated().sum()              # duplicatas

# 2. Análise univariada
df['col'].hist(bins=30)            # distribuição
df['col'].value_counts()           # categorias
df['col'].skew()                   # assimetria

# 3. Análise bivariada
df[['a','b']].corr()               # correlação
df.groupby('cat')['num'].mean()    # média por grupo
sns.scatterplot(data=df, x='a', y='b', hue='cat')

# 4. Análise categórica
pd.crosstab(df['cat1'], df['cat2'], normalize='index') * 100
df.pivot_table(values='num', index='cat1', columns='cat2', aggfunc='mean')

# 5. Storytelling
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
# ... montar painel narrativo com título principal e subtítulos informativos
```

---
*Próximo: **Projeto Final — Análise Exploratória Completa com Dataset Real***